# Підготовка датасету для виявлення фейкових медичних відгуків

Цей notebook реалізує методологію підготовки датасету згідно з розділом 3 звіту:
- Завантаження та очищення даних
- Попередня обробка текстів
- Балансування датасету 1:1 (5000 автентичних + 5000 фейкових)
- Перемішування датасету
- Збереження фінального датасету

## 1. Імпорти та налаштування

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Налаштування відображення pandas
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 10)

# Шляхи до файлів
DATA_DIR = Path('../../data')
DOCTOR_REVIEWS_PATH = DATA_DIR / 'doctors_reviews.csv'
FAKE_REVIEWS_PATH = DATA_DIR / 'fake_reviews.csv'
OUTPUT_PATH = DATA_DIR / 'final_dataset.csv'

# Параметри датасету
TARGET_REAL_REVIEWS = 5000
TARGET_FAKE_REVIEWS = 5000
MIN_TEXT_LENGTH = 50  # мінімальна довжина тексту в символах
RANDOM_STATE = 42

print(f"Робоча директорія: {Path.cwd()}")
print(f"Цільовий розмір датасету: {TARGET_REAL_REVIEWS + TARGET_FAKE_REVIEWS:,} відгуків (1:1)")

## 2. Функції попередньої обробки

очищення тексту, нормалізація, заміна російських літер, видаленя ієрогліфів, нормалізація пробілів

In [ ]:
def clean_text(text: str) -> str:
    """
    Очищення тексту від HTML-тегів, спеціальних символів та нормалізація.
    Згідно з методологією з розділу 3.3 звіту.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Видалення пустих або майже пустих рядків
    text = text.strip()
    if not text or text.lower() in ['nan', 'none', 'null', '']:
        return ""
    
    # Видалення HTML-тегів
    text = re.sub(r'<[^>]+>', '', text)
    
    # Видалення URL
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Видалення китайських, японських та корейських ієрогліфів
    # CJK Unified Ideographs
    text = re.sub(r'[\u4E00-\u9FFF]+', '', text)
    # CJK Extension A
    text = re.sub(r'[\u3400-\u4DBF]+', '', text)
    # CJK Compatibility Ideographs
    text = re.sub(r'[\uF900-\uFAFF]+', '', text)
    # Hiragana and Katakana
    text = re.sub(r'[\u3040-\u309F\u30A0-\u30FF]+', '', text)
    # Hangul (Korean)
    text = re.sub(r'[\uAC00-\uD7AF]+', '', text)
    
    # Нормалізація пробілів та розділових знаків
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([.,!?;:])', r'\1', text)
    
    # Заміна російських літер на українські аналоги
    replacements = {
        'ы': 'и', 'Ы': 'И',
        'ё': 'е', 'Ё': 'Е',
        'э': 'е', 'Э': 'Е',
        'ъ': "'", 'Ъ': "'"
    }
    for rus, ukr in replacements.items():
        text = text.replace(rus, ukr)
    
    # Видалення зайвих пробілів на початку та в кінці
    text = text.strip()
    
    return text


def is_valid_text(text: str, min_length: int = MIN_TEXT_LENGTH) -> bool:
    """
    Перевірка валідності тексту: мінімальна довжина та наявність українських літер.
    """
    if not text or len(text) < min_length:
        return False
    
    if not re.search(r'[а-яА-ЯіїєґІЇЄҐ]', text):
        return False
    
    return True




def remove_latin_chars(text: str) -> str:
    """
    Видалення латинських літер з тексту.
    Використовується для очищення фейкових відгуків від артефактів генерації.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Видалення латинських літер (a-z, A-Z)
    text = re.sub(r'[a-zA-Z]+', '', text)
    
    # Нормалізація пробілів після видалення
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


print("Функції попередньої обробки визначено (включаючи видалення латинських літер)")

## 3. Завантаження даних

In [ ]:
# Перевірка наявності файлів
if not DOCTOR_REVIEWS_PATH.exists():
    raise FileNotFoundError(f"Файл не знайдено: {DOCTOR_REVIEWS_PATH}")
if not FAKE_REVIEWS_PATH.exists():
    raise FileNotFoundError(f"Файл не знайдено: {FAKE_REVIEWS_PATH}")

print("Завантаження автентичних відгуків...")
df_real = pd.read_csv(DOCTOR_REVIEWS_PATH)
print(f"Завантажено: {len(df_real):,} рядків")
print(f"Колонки: {df_real.columns.tolist()}")
print(f"\nПерші записи:")
display(df_real.head(3))

print("\n" + "="*80)
print("Завантаження фейкових відгуків...")
df_fake = pd.read_csv(FAKE_REVIEWS_PATH)
print(f"Завантажено: {len(df_fake):,} рядків")
print(f"Колонки: {df_fake.columns.tolist()}")
print(f"\nПерші записи:")
display(df_fake.head(3))

## 4. Попередня обробка автентичних відгуків

In [ ]:
print("Обробка автентичних відгуків...\n")

# Вибір колонки з текстом відгуку
text_column = 'Коментар' if 'Коментар' in df_real.columns else 'text'
df_real['text'] = df_real[text_column].astype(str)

print(f"Початкова кількість: {len(df_real):,}")

# Очищення тексту
df_real['text'] = df_real['text'].apply(clean_text)
print(f"Після очищення: {len(df_real):,}")

# Фільтрація за валідністю
df_real['is_valid'] = df_real['text'].apply(is_valid_text)
df_real = df_real[df_real['is_valid']].copy()
print(f"Після фільтрації (min {MIN_TEXT_LENGTH} символів): {len(df_real):,}")

# Видалення дублікатів
df_real = df_real.drop_duplicates(subset=['text'], keep='first')
print(f"Після видалення дублікатів: {len(df_real):,}")

# Додавання флагу
df_real['fake'] = 0

# Вибір необхідної кількості
if len(df_real) > TARGET_REAL_REVIEWS:
    df_real = df_real.sample(n=TARGET_REAL_REVIEWS, random_state=RANDOM_STATE)
    print(f"\nВипадкова вибірка {TARGET_REAL_REVIEWS:,} відгуків")
elif len(df_real) < TARGET_REAL_REVIEWS:
    print(f"\nдоступно лише {len(df_real):,} автентичних відгуків (потрібно {TARGET_REAL_REVIEWS:,})")
    print(f"Буде використано всі {len(df_real):,} відгуків")

# Залишити тільки потрібні колонки
df_real = df_real[['text', 'fake']].reset_index(drop=True)

print(f"\nФінальна кількість автентичних відгуків: {len(df_real):,}")
print(f"Середня довжина тексту: {df_real['text'].str.len().mean():.1f} символів")
print(f"\nПриклади:")
display(df_real.head(3))

## 5. Попередня обробка фейкових відгуків

In [ ]:
print("Обробка фейкових відгуків...\n")

# Вибір колонки з текстом
df_fake['text'] = df_fake['text'].astype(str)

print(f"Початкова кількість: {len(df_fake):,}")

# Очищення тексту
df_fake['text'] = df_fake['text'].apply(clean_text)

# Видалення латинських літер (артефакти генерації)
df_fake['text'] = df_fake['text'].apply(remove_latin_chars)
print(f"Після очищення: {len(df_fake):,}")

# Фільтрація за валідністю
df_fake['is_valid'] = df_fake['text'].apply(is_valid_text)
df_fake = df_fake[df_fake['is_valid']].copy()
print(f"Після фільтрації (min {MIN_TEXT_LENGTH} символів): {len(df_fake):,}")

# Видалення дублікатів
df_fake = df_fake.drop_duplicates(subset=['text'], keep='first')
print(f"Після видалення дублікатів: {len(df_fake):,}")

# Додавання флагу
df_fake['fake'] = 1

# Вибір необхідної кількості
if len(df_fake) > TARGET_FAKE_REVIEWS:
    df_fake = df_fake.sample(n=TARGET_FAKE_REVIEWS, random_state=RANDOM_STATE)
    print(f"\nВипадкова вибірка {TARGET_FAKE_REVIEWS:,} відгуків")
elif len(df_fake) < TARGET_FAKE_REVIEWS:
    print(f"\nдоступно лише {len(df_fake):,} фейкових відгуків (потрібно {TARGET_FAKE_REVIEWS:,})")
    print(f"Буде використано всі {len(df_fake):,} відгуків")

# Залишити тільки потрібні колонки
df_fake = df_fake[['text', 'fake']].reset_index(drop=True)

print(f"\nФінальна кількість фейкових відгуків: {len(df_fake):,}")
print(f"Середня довжина тексту: {df_fake['text'].str.len().mean():.1f} символів")
print(f"\nПриклади:")
display(df_fake.head(3))

## 6. Об'єднання та перемішування датасету

In [ ]:
# Об'єднання автентичних та фейкових відгуків
df_combined = pd.concat([df_real, df_fake], ignore_index=True)

# перемішування датасету
# Використовуємо фіксований random_state для відтворюваності результатів
df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Додаткове перемішування для впевненості
df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE + 1).reset_index(drop=True)

print("Фінальний датасет:")
print(f"Загальна кількість: {len(df_combined):,} відгуків")
print(f"\nРозподіл за класами:")
class_distribution = df_combined['fake'].value_counts().sort_index()
print(f"Автентичні (fake=0): {class_distribution[0]:,} ({class_distribution[0]/len(df_combined)*100:.1f}%)")
print(f"Фейкові (fake=1): {class_distribution[1]:,} ({class_distribution[1]/len(df_combined)*100:.1f}%)")

print(f"\nСтатистика довжини тексту:")
text_lengths = df_combined['text'].str.len()
print(f"Мінімальна: {text_lengths.min()} символів")
print(f"Максимальна: {text_lengths.max()} символів")
print(f"Середня: {text_lengths.mean():.1f} символів")
print(f"Медіана: {text_lengths.median():.1f} символів")

print(f"\nПерші 10 записів після перемішування:")
display(df_combined.head(10))

## 7. Збереження фінального датасету

In [ ]:
# Збереження повного датасету
df_combined.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f"Датасет збережено: {OUTPUT_PATH}")
print(f"Розмір файлу: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")

print(f"\n{'='*60}")
print("Підготовка датасету завершена")
print(f"\nФінальний датасет:")
print(f"Загальна кількість: {len(df_combined):,} відгуків")
print(f"Автентичні: {(df_combined['fake']==0).sum():,} ({(df_combined['fake']==0).sum()/len(df_combined)*100:.1f}%)")
print(f"Фейкові: {(df_combined['fake']==1).sum():,} ({(df_combined['fake']==1).sum()/len(df_combined)*100:.1f}%)")

## 8. Аналіз та візуалізація датасету

In [ ]:
# Налаштування візуалізації
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Розподіл класів
class_counts = df_combined['fake'].value_counts()
axes[0].bar(['Автентичні', 'Фейкові'], class_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Розподіл класів', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Кількість відгуків')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')

# 2. Розподіл довжини тексту
df_combined['text_length'] = df_combined['text'].str.len()
axes[1].hist([df_combined[df_combined['fake']==0]['text_length'],
              df_combined[df_combined['fake']==1]['text_length']],
             label=['Автентичні', 'Фейкові'], bins=30, alpha=0.7, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Розподіл довжини текстів', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Довжина (символів)')
axes[1].set_ylabel('Кількість')
axes[1].legend()

# 3. Box plot довжини


plt.tight_layout()
plt.show()

# Детальна статистика
print("\nДетальна статистика по класах:")
print("\nАвтентичні відгуки:")
print(df_combined[df_combined['fake']==0]['text_length'].describe())
print("\nФейкові відгуки:")
print(df_combined[df_combined['fake']==1]['text_length'].describe())